# 🚀 Entrenamiento de LoRA: Claire (Lost Sword)
### Modelo Base: Illustrious-XL v0.1 (SDXL Anime) | Optimizado para GPU T4 en Colab

**Pasos para entrenar:**
1. Sube el archivo `dataset_claire.zip` (o `dataset.zip`) al panel de la izquierda.
2. Ejecuta cada celda en orden con el botón de Play (▶️).

In [ ]:
# @title 1. ⚙️ Instalar Dependencias y Parche de Compatibilidad Kohya
!pip install -q --upgrade pip
!pip install -q --upgrade diffusers transformers huggingface_hub accelerate bitsandbytes
!pip install -q toml safetensors ftfy torchvision xformers torchao voluptuous imagesize einops

import os
if not os.path.exists('/content/sd-scripts'):
    print('Clonando sd-scripts de Kohya...')
    !git clone https://github.com/kohya-ss/sd-scripts.git /content/sd-scripts

# Parche automático para CLIPTextModel en transformers modernos (Python 3.13 / PyTorch 2.6)
filepath = '/content/sd-scripts/library/sdxl_model_util.py'
if os.path.exists(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        code = f.read()
    patch_func = '''def _load_state_dict_on_device(model, state_dict, device=None, dtype=None):
    model_sd = model.state_dict()
    model_keys = set(model_sd.keys())
    adapted_sd = {}
    for k, v in state_dict.items():
        if k in model_keys:
            adapted_sd[k] = v
        elif k.startswith("text_model.") and k[11:] in model_keys:
            adapted_sd[k[11:]] = v
        elif ("text_model." + k) in model_keys:
            adapted_sd["text_model." + k] = v
        else:
            adapted_sd[k] = v
    try:
        info = model.load_state_dict(adapted_sd, strict=False, assign=True)
    except TypeError:
        info = model.load_state_dict(adapted_sd, strict=False)
    if device is not None:
        model.to(device)
    if dtype is not None:
        model.to(dtype)
    return info'''
    start = code.find('def _load_state_dict_on_device(')
    end = code.find('def load_models_from_sdxl_checkpoint(')
    if start != -1 and end != -1:
        code = code[:start] + patch_func + '\n\n' + code[end:]
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(code)
        print('✓ Parche de compatibilidad CLIPTextModel y meta-tensors aplicado a sd-scripts.')
    else:
        print('Nota: Verificando estructura de sd-scripts...')
print('✓ Entorno y sd-scripts 100% listos para entrenar.')

In [ ]:
# @title 2. 📥 Descargar Modelo Base Illustrious-XL
import os
from huggingface_hub import hf_hub_download

MODELS_DIR = '/content/models'
os.makedirs(MODELS_DIR, exist_ok=True)
LOCAL_MODEL_PATH = os.path.join(MODELS_DIR, 'Illustrious-XL-v0.1.safetensors')

if not os.path.exists(LOCAL_MODEL_PATH):
    print('Descargando Illustrious-XL v0.1 (~6.6 GB)...')
    LOCAL_MODEL_PATH = hf_hub_download(
        repo_id='OnomaAIResearch/Illustrious-xl-early-release-v0',
        filename='Illustrious-XL-v0.1.safetensors',
        local_dir=MODELS_DIR
    )
print('✓ Modelo base listo en:', LOCAL_MODEL_PATH)

In [ ]:
# @title 3. 📦 Extraer Dataset y Generar dataset.toml para Claire (Lost Sword)
import os, shutil, zipfile, glob, toml

CONCEPT_NAME = 'claire'
REPEATS = 8
CLASS_TOKENS = 'claire_(lost_sword), 1girl, nun'
LORA_OUTPUT_NAME = 'claire_lost_sword_lora'

DATASET_DIR = '/content/dataset'
OUTPUT_DIR = '/content/output_lora'
TARGET_DIR = f'/content/dataset/{REPEATS}_{CONCEPT_NAME}'

if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
os.makedirs(TARGET_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

posibles_zips = ['/content/dataset_claire.zip', '/content/dataset.zip', '/content/drive/MyDrive/dataset_claire.zip', '/content/drive/MyDrive/dataset.zip']
zip_encontrado = None
for p in posibles_zips:
    if os.path.exists(p):
        zip_encontrado = p
        break

if not zip_encontrado:
    buscados = glob.glob('/content/dataset*.zip') + glob.glob('/content/*.zip')
    if buscados:
        zip_encontrado = buscados[0]

if zip_encontrado and os.path.exists(zip_encontrado):
    print(f'Descomprimiendo {zip_encontrado} en {TARGET_DIR}...')
    with zipfile.ZipFile(zip_encontrado, 'r') as z:
        z.extractall(TARGET_DIR)
    imgs = [f for f in os.listdir(TARGET_DIR) if f.endswith(('.png', '.jpg', '.webp'))]
    txts = [f for f in os.listdir(TARGET_DIR) if f.endswith('.txt')]
    print(f'✅ Dataset listo: {len(imgs)} imágenes y {len(txts)} etiquetas (.txt)')
else:
    print('⚠️ No se encontró dataset_claire.zip. Sube el archivo ZIP a Colab.')

# Generar dataset.toml
dataset_config = {
    'general': {
        'enable_bucket': True,
        'caption_extension': '.txt',
        'shuffle_caption': False,
        'bucket_reso_steps': 64,
        'min_bucket_reso': 512,
        'max_bucket_reso': 2048
    },
    'datasets': [
        {
            'resolution': 1024,
            'min_bucket_reso': 512,
            'max_bucket_reso': 2048,
            'caption_dropout_rate': 0.0,
            'subsets': [
                {
                    'image_dir': TARGET_DIR,
                    'num_repeats': REPEATS,
                    'class_tokens': CLASS_TOKENS
                }
            ]
        }
    ]
}

with open('/content/dataset.toml', 'w') as f:
    toml.dump(dataset_config, f)
print('✓ Archivo /content/dataset.toml configurado.')

In [ ]:
# @title 4. 🚀 Entrenar LoRA (Claire (Lost Sword))
import subprocess, sys, os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

train_cmd = f"""python /content/sd-scripts/sdxl_train_network.py \
  --pretrained_model_name_or_path="{LOCAL_MODEL_PATH}" \
  --dataset_config="/content/dataset.toml" \
  --output_dir="{OUTPUT_DIR}" \
  --output_name="claire_lost_sword_lora" \
  --save_model_as="safetensors" \
  --network_module="networks.lora" \
  --network_dim=16 \
  --network_alpha=8 \
  --network_train_unet_only \
  --train_batch_size=1 \
  --max_train_epochs=10 \
  --learning_rate=1e-4 \
  --unet_lr=1e-4 \
  --optimizer_type="AdamW8bit" \
  --lr_scheduler="cosine" \
  --mixed_precision="fp16" \
  --save_precision="fp16" \
  --gradient_checkpointing \
  --cache_latents \
  --cache_latents_to_disk \
  --cache_text_encoder_outputs \
  --cache_text_encoder_outputs_to_disk \
  --fp8_base \
  --vae_batch_size=1 \
  --xformers \
  --lowram \
  --save_every_n_epochs=2 \
  --seed=42"""

print('🚀 Iniciando entrenamiento de Claire (Lost Sword)...')
process = subprocess.Popen(train_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in iter(process.stdout.readline, ''):
    print(line, end='')
    sys.stdout.flush()
process.wait()
print(f'\n[FIN] Código de salida: {process.returncode}')

In [ ]:
# @title 5. 💾 Guardar LoRAs en Google Drive
import shutil, os
from google.colab import drive

try:
    drive.mount('/content/drive')
    DRIVE_SAVE_PATH = '/content/drive/MyDrive/Illustrious_LoRAs'
    os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
    for f in os.listdir(OUTPUT_DIR):
        if f.endswith('.safetensors'):
            src = os.path.join(OUTPUT_DIR, f)
            dst = os.path.join(DRIVE_SAVE_PATH, f)
            shutil.copy2(src, dst)
            print(f'✓ Copiado a Google Drive: {dst}')
except Exception as e:
    print('Google Drive:', e)
    print('Tus LoRAs están en:', OUTPUT_DIR)

In [ ]:
!pip install -q --upgrade torchao

# @title 6. 🎨 Prueba de Inferencia del LoRA
import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

LORA_FILE = os.path.join(OUTPUT_DIR, 'claire_lost_sword_lora.safetensors')
if not os.path.exists(LORA_FILE) and os.path.exists(OUTPUT_DIR):
    safes = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.safetensors')]
    if safes:
        LORA_FILE = os.path.join(OUTPUT_DIR, sorted(safes)[-1])

if os.path.exists(LORA_FILE):
    print('Cargando pipeline SDXL con Illustrious...')
    pipe = StableDiffusionXLPipeline.from_single_file(LOCAL_MODEL_PATH, torch_dtype=torch.float16, use_safetensors=True)
    pipe.vae.enable_slicing()
    pipe.vae.enable_tiling()
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    pipe.load_lora_weights(LORA_FILE, weight_name=os.path.basename(LORA_FILE))
    prompt = 'masterpiece, best quality, 1girl, claire_(lost_sword), solo, gray hair, long hair, blindfold, not visible eyes, nun, veil, white veil, nun habit, holding staff, standing, official art, detailed illustration'
    negative_prompt = '(holding weapon:1.4), worst quality, low quality, bad anatomy, bad hands, blurry, watermark, signature'
    image = pipe(prompt=prompt, negative_prompt=negative_prompt, width=1024, height=1024, num_inference_steps=28, guidance_scale=6.5).images[0]
    image.save('/content/test_inference.png')
    display(image)
else:
    print('No se encontró el archivo LoRA para probar.')



